# GitHub Actions Importer - Quick Reference Card

## Installation & Setup



In [ ]:
# Install the CLI extension
gh extension install github/gh-actions-importer

# Update to latest version
gh actions-importer update

# Configure credentials (interactive)
gh actions-importer configure

# Check version
gh actions-importer version



## Environment Variables (Alternative to Interactive Config)



In [ ]:
export GITHUB_ACCESS_TOKEN="ghp_xxxxx"
export GITHUB_INSTANCE_URL="https://github.com"

# Jenkins
export JENKINS_ACCESS_TOKEN="xxxxx"
export JENKINS_USERNAME="admin"
export JENKINS_INSTANCE_URL="https://jenkins.example.com"



## The 4-Phase Migration Workflow

### Phase 1: AUDIT (Analyze Current State)



In [ ]:
# Audit entire Jenkins instance
gh actions-importer audit jenkins \
  --output-dir ./audit-results \
  --jenkins-url https://jenkins.example.com \
  --jenkins-username admin \
  --jenkins-access-token $JENKINS_TOKEN

# What you get:
# - audit_summary.md (high-level overview)
# - jobs.json (complete inventory)
# - plugins.json (plugin usage analysis)
# - shared_libraries.json (dependencies)



**Output Interpretation:**


In [ ]:
✅ Total Pipelines: 847
✅ Simple (auto-convert): 512 (60%)
⚠️  Medium (minor fixes): 254 (30%)
❌ Complex (manual work): 81 (10%)
📊 Conversion Rate: 78%



### Phase 2: FORECAST (Predict Costs & Capacity)



In [ ]:
# Forecast GitHub Actions usage
gh actions-importer forecast jenkins \
  --output-dir ./forecast \
  --jenkins-url https://jenkins.example.com \
  --jenkins-username admin \
  --jenkins-access-token $JENKINS_TOKEN \
  --start-date 2025-01-01 \
  --end-date 2025-02-01

# What you get:
# - Estimated Actions minutes/month
# - Peak concurrent jobs
# - Runner recommendations
# - Cost estimates



**Key Metrics:**


In [ ]:
Minutes/Month: 1,234,567
Concurrent Jobs (p95): 87
Estimated Cost: $X,XXX/month
Recommended Runners:
  - Self-hosted (small): 40
  - Self-hosted (medium): 30
  - Self-hosted (large): 10



### Phase 3: DRY-RUN (Test Conversion)



In [ ]:
# Convert single pipeline (local output only)
gh actions-importer dry-run jenkins \
  --source-url https://jenkins.example.com/job/my-project/job/build \
  --output-dir ./converted \
  --jenkins-username admin \
  --jenkins-access-token $JENKINS_TOKEN

# What you get:
# - Converted workflow YAML
# - Conversion report (warnings)
# - Manual steps needed
# - Side-by-side comparison



**Review Checklist:**


In [ ]:
- [ ] All steps converted?
- [ ] Secrets properly referenced?
- [ ] Conditions correct?
- [ ] Matrix strategy needed?
- [ ] Custom actions required?



### Phase 4: MIGRATE (Create PR)



In [ ]:
# Convert and create pull request
gh actions-importer migrate jenkins \
  --source-url https://jenkins.example.com/job/my-project/job/build \
  --target-url https://github.com/example/my-project \
  --output-dir ./migration \
  --jenkins-username admin \
  --jenkins-access-token $JENKINS_TOKEN \
  --github-access-token $GITHUB_PAT

# What you get:
# - Automatic PR creation
# - Workflow file in .github/workflows/
# - Migration notes in PR description
# - Checklist of manual steps



## Batch Operations

### Audit Multiple Jobs


In [ ]:
# Create list of jobs to audit
cat > jobs.txt <<EOF
https://jenkins.example.com/job/team-a/job/build
https://jenkins.example.com/job/team-a/job/test
https://jenkins.example.com/job/team-a/job/deploy
EOF

# Batch audit
while read job_url; do
  gh actions-importer dry-run jenkins \
    --source-url "$job_url" \
    --output-dir "./converted/$(basename $job_url)"
done &lt; jobs.txt



### Migrate by Pattern


In [ ]:
# Find all maven builds
gh actions-importer audit jenkins --json | \
  jq -r '.pipelines[] | select(.type=="maven") | .url' | \
while read url; do
  echo "Migrating: $url"
  gh actions-importer migrate jenkins --source-url "$url"
done



## IssueOps (Self-Service Migration)



In [ ]:
# Clone the IssueOps template
gh repo create example/migration-ops \
  --template actions/importer-issue-ops \
  --private

# Users create issues with:
# - Jenkins job URL
# - Target repo
# - Team mention

# Workflow auto-creates migration PRs



**Issue Template Example:**


In [ ]:
name: Migrate Jenkins Pipeline
labels: [migrate-jenkins]

Jenkins URL: https://jenkins.example.com/job/my-pipeline
Target Repo: example/my-project
Team: @example/backend-team



## Advanced Options

### Custom Transformers


In [ ]:
# Use custom transformer for XYZ-specific patterns
gh actions-importer migrate jenkins \
  --source-url https://jenkins.amd.com/job/build \
  --custom-transformers ./transformers/amd-custom.rb



### Specific Job Types


In [ ]:
# Multibranch pipelines
gh actions-importer dry-run jenkins \
  --source-url https://jenkins.example.com/job/multibranch-project

# Freestyle jobs
gh actions-importer dry-run jenkins \
  --source-url https://jenkins.example.com/job/freestyle-job

# Pipeline jobs
gh actions-importer dry-run jenkins \
  --source-url https://jenkins.example.com/job/pipeline-job



### Organization/Folder Level


In [ ]:
# Migrate entire folder
gh actions-importer audit jenkins \
  --source-url https://jenkins.example.com/job/team-folder \
  --output-dir ./team-folder-audit



## Troubleshooting

### Common Issues

**1. Authentication Errors**


In [ ]:
# Verify credentials
curl -u $JENKINS_USERNAME:$JENKINS_TOKEN \
  https://jenkins.example.com/api/json

# For GitHub
gh auth status



**2. Plugin Not Supported**


In [ ]:
# Check warnings in dry-run output
# Look for: "TODO: Manual migration required"
# Create custom action or use alternative



**3. Shared Library Conversion**


In [ ]:
# Shared libraries need manual conversion
# Strategy: Convert to composite actions or reusable workflows
# See: .github/actions/ or .github/workflows/reusable-*.yml



**4. Complex Groovy Logic**


In [ ]:
# Extract Groovy scripts
# Convert to shell scripts or Python
# Use composite actions or custom actions



## Success Metrics

### Track Migration Progress


In [ ]:
#!/bin/bash
# migration-metrics.sh

echo "=== Migration Dashboard ==="
echo "Total Pipelines: $(jq '.total' audit-results/audit_summary.json)"
echo "Migrated: $(gh api repos/example/my-project/workflows -q '.workflows | length')"
echo "In Progress: $(gh pr list --label jenkins-migration --json number | jq 'length')"
echo "Success Rate: 78%"
echo "Time Saved: $(( (512 * 8) - (512 * 2) )) hours"



### Validation Checklist


In [ ]:
## Post-Migration Validation

- [ ] Workflow triggers correctly (push/PR/schedule)
- [ ] All build steps execute
- [ ] Tests pass with same coverage
- [ ] Artifacts generated correctly
- [ ] Secrets/variables configured
- [ ] Notifications working
- [ ] Build time within 20% of Jenkins
- [ ] Team trained on new workflow



## ROI Calculator



In [ ]:
#!/bin/bash
# Calculate savings

TOTAL_PIPELINES=500
HOURS_PER_MANUAL=8
HOURS_PER_AUTOMATED=2
ENGINEER_RATE=150

MANUAL_COST=$(( TOTAL_PIPELINES * HOURS_PER_MANUAL * ENGINEER_RATE ))
AUTO_COST=$(( TOTAL_PIPELINES * HOURS_PER_AUTOMATED * ENGINEER_RATE ))
SAVINGS=$(( MANUAL_COST - AUTO_COST ))

echo "Manual Migration: \$$MANUAL_COST"
echo "With Importer: \$$AUTO_COST"
echo "Savings: \$$SAVINGS (75% reduction)"
echo "Time Saved: $(( TOTAL_PIPELINES * (HOURS_PER_MANUAL - HOURS_PER_AUTOMATED) )) hours"



## Quick Commands Cheat Sheet

| Task | Command |
|------|---------|
| **Install** | `gh extension install github/gh-actions-importer` |
| **Update** | `gh actions-importer update` |
| **Configure** | `gh actions-importer configure` |
| **Audit All** | `gh actions-importer audit jenkins` |
| **Forecast** | `gh actions-importer forecast jenkins` |
| **Test Convert** | `gh actions-importer dry-run jenkins --source-url <URL>` |
| **Migrate (PR)** | `gh actions-importer migrate jenkins --source-url <URL>` |
| **Version** | `gh actions-importer version` |



## Resources

- **Official Docs**: https://docs.github.com/en/actions/migrating-to-github-actions/automated-migrations
- **Labs Repository**: https://github.com/actions/importer-labs
- **IssueOps Template**: https://github.com/actions/importer-issue-ops
- **GitHub CLI**: https://cli.github.com/

---
